# I22 batch processing with glassy-carbon absolute calibration

This notebook is the ordinary, non-chunked I22 workflow. It preprocesses only incompatible frame metadata, derives one SAXS absolute-intensity factor from glassy-carbon measurement 966700 and the supplied reference curve, then applies that factor to parallel SAXS and WAXS correction lanes in one graph. The two corrected 1D curves are concatenated, Q-sorted, and exposed as one dataset and plot. Geometry and detector-specific corrections remain in the tracked MoDaCor pipeline.

The three chunk-transport examples live in separate notebooks beside this one. Run from top to bottom and edit only **Configuration** for normal use.


## Environment

Use a kernel containing the current MoDaCor checkout with the `server`, `attenuation`, and `plotting` extras plus `hdf5plugin` and `matplotlib`.


In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "example_utils.py").is_file():
        EXAMPLES_ROOT = candidate
        break
else:
    raise FileNotFoundError("Start Jupyter from MoDaCor_examples or one of its subdirectories.")

sys.path.insert(0, str(EXAMPLES_ROOT))
from example_utils import locate_example_dir

PROJECT_DIR = locate_example_dir("DLS/I22")
sys.path.insert(0, str(PROJECT_DIR))

import atexit
import sys

from IPython.display import JSON, Markdown, display

import hdf5plugin
import h5py
import modacor
from modacor.client import LocalRuntimeServer
from modacor.runner.pipeline import Pipeline

from i22_helpers import (
    prepare_inputs,
    source_registrations,
)


## Configuration


In [ ]:
MAX_MEASUREMENTS = None  # use 1 for a quick smoke run
BEAMLINE_CONFIGURATION = "usaxs_saxs_waxs"  # special nosecone; no aluminium attenuator
# Unobstructed-beam measurement used to calibrate bsdiodes against I0.
TRANSMISSION_REFERENCE_FILE = PROJECT_DIR / "data" / "i22-977723.nxs"
INTENSITY_CALIBRATION_FILE = PROJECT_DIR / "data" / "i22-966700.nxs"
INTENSITY_CALIBRATION_REFERENCE_FILE = PROJECT_DIR / "data" / "Glassy Carbon L average.dat"
INTENSITY_CALIBRATION_THICKNESS = 1.0
INTENSITY_CALIBRATION_THICKNESS_UNITS = "mm"
SAMPLE_THICKNESS = 50e-6  # m; from the acquisition title: 50 micron 3YSZ-PEGDA
SAMPLE_THICKNESS_UNITS = "m"
OVERWRITE_PREPROCESSED = False

SERVER_HOST = "127.0.0.1"
SERVER_PORT = 8901
TRACE = {"enabled": True, "watch": {"sample": ["signal"], "waxs_sample": ["signal"]}}
OUTPUT_DATA_PATHS = [
    "/saxs_waxs_1d/signal",
    "/saxs_waxs_1d/Q",
    "/saxs_waxs_1d/detector_index",
    "/intensity_calibration/signal",
    "/intensity_calibration/Q",
    "/intensity_calibration_fit/absolute_intensity_scale_factor",
    "/intensity_calibration_fit/absolute_intensity_fit_point_count",
    "/intensity_calibration_fit/absolute_intensity_fit_x_min",
    "/intensity_calibration_fit/absolute_intensity_fit_x_max",
    "/intensity_calibration_fit/absolute_intensity_fit_reduced_chi_square",
]
OUTPUT_DIR = PROJECT_DIR / "work" / "output"


## Prepare inputs

The helper discovers complete sample masters and writes compact files below `work/preprocessed/`. Detector images remain external links; both diode readouts are averaged and their SEMs are propagated through the unobstructed-beam calibration to `/entry1/sample/transmission` and `/entry1/sample/transmission_sem`. Count times are reshaped, the title-derived 50 µm sample thickness is recorded, and measurement 966700 is prepared as the reusable 1.0 mm glassy-carbon calibration source. The combined pipeline fits one factor in the conservative SAXS range and applies it to both detectors.


In [ ]:
inputs = prepare_inputs(
    PROJECT_DIR,
    beamline_configuration=BEAMLINE_CONFIGURATION,
    transmission_reference_file=TRANSMISSION_REFERENCE_FILE,
    intensity_calibration_file=INTENSITY_CALIBRATION_FILE,
    intensity_calibration_reference_file=INTENSITY_CALIBRATION_REFERENCE_FILE,
    intensity_calibration_thickness=INTENSITY_CALIBRATION_THICKNESS,
    intensity_calibration_thickness_units=INTENSITY_CALIBRATION_THICKNESS_UNITS,
    sample_thickness=SAMPLE_THICKNESS,
    sample_thickness_units=SAMPLE_THICKNESS_UNITS,
    overwrite=OVERWRITE_PREPROCESSED,
)

print(f"Python: {sys.executable}")
print(f"MoDaCor: {modacor.__version__}")
print(f"Measurements: {len(inputs.measurements)}")
print(f"Preprocessed data: {inputs.work_dir / 'preprocessed'}")


## Preview the correction graphs


In [ ]:
if inputs.combined_pipeline_path is None:
    raise ValueError(f"No combined pipeline is available for {BEAMLINE_CONFIGURATION!r}.")
pipeline = Pipeline.from_yaml_file(yaml_file=inputs.combined_pipeline_path)
pipeline.prepare()
display(Markdown(f"```mermaid\n{pipeline.to_mermaid(direction='TD')}\n```"))
print(f"Combined SAXS/WAXS graph: {len(pipeline.graph)} steps")


### Export the correction graph

This writes the grouped `for_each` graph as DOT, SVG, PDF, and native editable draw.io XML. Processing steps use rounded rectangular boxes. Graphviz must provide the `dot` executable.


In [ ]:
import shutil
import subprocess

if shutil.which("dot") is None:
    raise RuntimeError("Graphviz 'dot' is required to export the pipeline graph.")

GRAPH_OUTPUT_DIR = OUTPUT_DIR / "graphs"
GRAPH_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
graph_stem = inputs.combined_pipeline_path.stem
dot_path = GRAPH_OUTPUT_DIR / f"{graph_stem}.dot"
svg_path = GRAPH_OUTPUT_DIR / f"{graph_stem}.svg"
pdf_path = GRAPH_OUTPUT_DIR / f"{graph_stem}.pdf"
drawio_path = GRAPH_OUTPUT_DIR / f"{graph_stem}.drawio"
dot_lines = pipeline.to_dot(direction="TB").splitlines()
dot_lines.insert(1, '  node [shape=box, style="rounded"];')
dot_path.write_text("\n".join(dot_lines) + "\n", encoding="utf-8")
for output_format, output_path in (("svg", svg_path), ("pdf", pdf_path)):
    subprocess.run(
        ["dot", f"-T{output_format}", "-o", str(output_path), str(dot_path)],
        check=True,
    )
drawio_path.write_text(pipeline.to_drawio(direction="TB"), encoding="utf-8")

display(Markdown(
    "\n".join((
        f"- [SVG pipeline graph]({svg_path})",
        f"- [PDF pipeline graph]({pdf_path})",
        f"- [Editable draw.io pipeline graph]({drawio_path})",
        f"- DOT source: `{dot_path}`",
    ))
))


## Start or reuse a local runtime


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
server = LocalRuntimeServer(
    host=SERVER_HOST,
    port=SERVER_PORT,
    log_path=OUTPUT_DIR / "modacor_server.log",
    environment={"HDF5_PLUGIN_PATH": hdf5plugin.PLUGINS_PATH},
)
client = server.start()
atexit.register(server.stop)
print(f"{'Started' if server.launched else 'Reusing'} runtime at {client.base_url}")


## Process the batch

`mode="auto"` lets the server choose a full first run when no reusable state exists. Later sample changes reuse the unchanged background, geometry, mask, and glassy-carbon calibration branches. The SAXS-derived absolute scale is calculated once in the shared session and feeds both detector branches.


In [ ]:
measurements = inputs.measurements[:MAX_MEASUREMENTS]
results = []
session = client.replace_session(
    "i22-saxs-waxs-server-batch",
    name="I22 combined SAXS/WAXS server batch",
    pipeline_yaml_path=str(inputs.combined_pipeline_path),
    trace=TRACE,
)
session.register_sources(*source_registrations(inputs, sample=measurements[0][1]))
session.register_sink({"ref": "plots", "type": "plotly_json", "location": "buffer://session"})

for master, prepared in measurements:
    output = OUTPUT_DIR / f"{master.stem}_saxs_waxs_server_result.h5"
    session.register_source({"ref": "sample", "type": "hdf", "location": str(prepared)})
    session.register_sink({"ref": "result_hdf", "type": "hdf", "location": str(output)})
    run_name = f"{master.stem}_saxs_waxs"
    result = session.process(
        mode="auto",
        changed_sources=["sample"],
        run_name=run_name,
        rollback_snapshot=False,
        write_hdf={"path": str(output), "data_paths": OUTPUT_DATA_PATHS},
    )
    calibration_root = f"/processing/result/{run_name}/intensity_calibration_fit"
    with h5py.File(output, "r") as h5:
        scale = float(h5[f"{calibration_root}/absolute_intensity_scale_factor/signal"][()].item())
        reduced_chi_square = float(
            h5[f"{calibration_root}/absolute_intensity_fit_reduced_chi_square/signal"][()].item()
        )
    results.append({
        "measurement": master.name, "output": output,
        "absolute_intensity_scale_factor": scale,
        "absolute_intensity_fit_reduced_chi_square": reduced_chi_square, "result": result,
    })
    print(
        f"{master.name}: {result['status']} ({result['effective_mode']}), "
        f"shared dimensionless scale={scale:.6g}, reduced chi-square={reduced_chi_square:.4g}"
    )

print(f"Completed {len(results)} combined measurement runs.")


## Live plots and diagnostics


In [ ]:
links = [
    "## Latest runtime plots",
    f"- [Combined SAXS/WAXS corrected I(Q)]({session.plot_url('plots', 'saxs-waxs-1d')})",
    f"- [SAXS corrected detector image]({session.plot_url('plots', 'saxs-2d')})",
    f"- [WAXS corrected detector image]({session.plot_url('plots', 'waxs-2d')})",
]
display(Markdown("\n".join(links)))

display(JSON(session.latest_error()))


## Cleanup

This stops only a process launched by this notebook. A runtime that was already listening on the configured port is left alone.


In [ ]:
server.stop()
print("Stopped the notebook-owned runtime." if not client.is_ready() else "Left the external runtime running.")
